# 🌍 COVID-19 Global Analysis & Short-Term Forecasting

## Project Overview

This project analyzes reported COVID-19 case data at country and WHO-region level. The notebook follows a complete data-analysis workflow: data validation, cleaning, aggregation, exploratory analysis, epidemiological indicators, rolling trends, geographic visualization, and short-term forecasting.

### Analytical questions

- How did reported confirmed cases, deaths, and recoveries evolve globally?
- Which countries had the highest cumulative confirmed cases at the end of the observed period?
- How did reported cases differ across WHO regions?
- What do reported recovery and death rates look like over time?
- How does a 7-day rolling average change the interpretation of daily case counts?
- Can a short-term forecasting model reasonably reproduce the final portion of the observed cumulative case series?

> **Important:** These are reported surveillance data, not a direct measure of true infections. Reporting practices, testing capacity, definitions, and retrospective corrections varied across locations.

## 1. Setup

The analysis uses Python with Pandas for data manipulation, Matplotlib/Seaborn for static charts, Plotly for interactive charts, and Prophet for the forecasting section.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

The original notebook used a Google Drive CSV. The loader below first looks for a local copy in `data/`, then falls back to the original URL so the notebook remains usable in Colab.

For a public GitHub repository, keeping a local CSV under `data/` is preferable when the dataset's redistribution terms allow it.

In [ ]:
DATA_URL = "https://drive.google.com/uc?export=download&id=1-BQMLkROX5Ad47S7FixUN1E5T2HV-FML"
LOCAL_PATH = Path("data/covid_19.csv")

if LOCAL_PATH.exists():
    df_raw = pd.read_csv(LOCAL_PATH)
    data_source = f"Local file: {LOCAL_PATH}"
else:
    df_raw = pd.read_csv(DATA_URL)
    data_source = "Original Google Drive CSV URL"

print(f"Data source: {data_source}")
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 3. Understand the data

### Data dictionary

| Column | Description |
|---|---|
| `Province/State` | Province/state when reported |
| `Country/Region` | Country or territory |
| `Lat`, `Long` | Geographic coordinates |
| `Date` | Reporting date |
| `Confirmed` | Cumulative reported confirmed cases |
| `Deaths` | Cumulative reported deaths |
| `Recovered` | Cumulative reported recoveries |
| `Active` | Reported active cases |
| `WHO Region` | WHO geographic region |

In [ ]:
df_raw.info()

In [ ]:
missing_summary = (
    df_raw.isna()
    .sum()
    .to_frame("missing_values")
    .assign(missing_pct=lambda x: 100 * x["missing_values"] / len(df_raw))
    .sort_values("missing_values", ascending=False)
)
missing_summary

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("Date range:", df_raw["Date"].min(), "to", df_raw["Date"].max())
print("Countries/regions:", df_raw["Country/Region"].nunique())
print("WHO regions:", df_raw["WHO Region"].nunique())

## 4. Data cleaning and validation

The raw dataset contains province/state-level records for some locations. To make country-level comparisons consistent, province/state records are aggregated by country and date.

The cumulative measures are kept intact. Daily changes are calculated only after sorting within each country. Negative changes are **not silently clipped** because negative daily changes can represent retrospective corrections in reported cumulative data.

In [ ]:
df = df_raw.copy()

# Standardize date type and remove fields not required for the country-level analysis.
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.drop(columns=["Lat", "Long"], errors="ignore")

# Validate required fields.
required_columns = {
    "Country/Region", "Date", "Confirmed", "Deaths",
    "Recovered", "Active", "WHO Region"
}
missing_required = required_columns - set(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")

# Numeric columns should be numeric.
numeric_cols = ["Confirmed", "Deaths", "Recovered", "Active"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove exact duplicates before aggregation.
df = df.drop_duplicates().copy()

# Aggregate province/state observations to country-date level.
df = (
    df.groupby(["Country/Region", "Date"], as_index=False)
      .agg({
          "Confirmed": "sum",
          "Deaths": "sum",
          "Recovered": "sum",
          "Active": "sum",
          "WHO Region": "first"
      })
)

# Sort before calculating changes.
df = df.sort_values(["Country/Region", "Date"]).reset_index(drop=True)

print("Cleaned shape:", df.shape)
df.head()

In [ ]:
# Check for impossible missing values after cleaning.
print("Missing values after cleaning:")
print(df.isna().sum())

print("\nNegative cumulative values:")
print((df[numeric_cols] < 0).sum())

## 5. Daily changes and data-quality checks

Daily changes are derived from cumulative country-level totals. The first observation for each country has no previous day, so its daily change is set to zero for analytical convenience.

Negative changes are retained and counted rather than replaced with zero. This preserves information about revisions in the reporting series.

In [ ]:
for col in ["Confirmed", "Deaths", "Recovered"]:
    new_col = f"New_{col}"
    df[new_col] = df.groupby("Country/Region")[col].diff()
    df[new_col] = df[new_col].fillna(0)

negative_changes = {
    col: int((df[col] < 0).sum())
    for col in ["New_Confirmed", "New_Deaths", "New_Recovered"]
}
negative_changes

In [ ]:
# Inspect the rows affected by negative revisions.
revision_rows = df[
    (df["New_Confirmed"] < 0) |
    (df["New_Deaths"] < 0) |
    (df["New_Recovered"] < 0)
].copy()

revision_rows[[
    "Country/Region", "Date", "Confirmed", "New_Confirmed",
    "Deaths", "New_Deaths", "Recovered", "New_Recovered"
]].head(20)

## 6. Global trends

Global totals are calculated by summing country-level observations for each date. Because the source contains reported cumulative counts, these totals should be interpreted as reported counts rather than estimates of true infections.

In [ ]:
df_global = (
    df.groupby("Date", as_index=False)
      .agg({
          "Confirmed": "sum",
          "Deaths": "sum",
          "Recovered": "sum",
          "Active": "sum",
          "New_Confirmed": "sum",
          "New_Deaths": "sum",
          "New_Recovered": "sum"
      })
      .sort_values("Date")
      .reset_index(drop=True)
)

df_global.head()

In [ ]:
fig = px.line(
    df_global,
    x="Date",
    y=["Confirmed", "Deaths", "Recovered"],
    title="Global Reported COVID-19 Totals Over Time",
    labels={"value": "Reported cases", "variable": "Metric", "Date": "Date"}
)
fig.update_layout(legend_title="Metric")
fig.show()

In [ ]:
fig = px.line(
    df_global,
    x="Date",
    y=["New_Confirmed", "New_Deaths", "New_Recovered"],
    title="Global Daily Reported Changes",
    labels={"value": "Daily change", "variable": "Metric", "Date": "Date"}
)
fig.show()

### Key interpretation

Cumulative totals answer **how much has been reported so far**, while daily changes show **how quickly the reported totals are changing**. They should not be interpreted interchangeably.

## 7. Country-level analysis

The final observed date is used for the country comparison so that countries are compared at the same point in the dataset.

In [ ]:
latest_date = df["Date"].max()
latest = (
    df[df["Date"] == latest_date]
    .sort_values("Confirmed", ascending=False)
    .reset_index(drop=True)
)

print("Latest observed date:", latest_date.date())
latest[["Country/Region", "Confirmed", "Deaths", "Recovered", "Active"]].head(10)

In [ ]:
top10 = latest.head(10).sort_values("Confirmed")

fig = px.bar(
    top10,
    x="Confirmed",
    y="Country/Region",
    orientation="h",
    title=f"Top 10 Countries by Reported Confirmed Cases — {latest_date:%d %b %Y}",
    labels={"Confirmed": "Confirmed cases", "Country/Region": "Country/Region"}
)
fig.show()

## 8. Epidemiological indicators

Two descriptive indicators are calculated from cumulative reported values:

- **Reported death rate** = cumulative deaths / cumulative confirmed cases
- **Reported recovery rate** = cumulative recovered / cumulative confirmed cases

These are descriptive ratios within this dataset. They should not be interpreted as a clinical case-fatality rate or individual probability of recovery.

In [ ]:
# Correct calculation: use the global totals, not the country-level dataframe.
df_global["Death_rate"] = np.where(
    df_global["Confirmed"] > 0,
    df_global["Deaths"] / df_global["Confirmed"],
    0
)

df_global["Recovery_rate"] = np.where(
    df_global["Confirmed"] > 0,
    df_global["Recovered"] / df_global["Confirmed"],
    0
)

indicator_summary = df_global[["Date", "Death_rate", "Recovery_rate"]].tail()
indicator_summary

In [ ]:
fig = px.line(
    df_global,
    x="Date",
    y=["Recovery_rate", "Death_rate"],
    title="Global Reported Recovery and Death Ratios",
    labels={"value": "Ratio", "variable": "Indicator", "Date": "Date"}
)
fig.update_yaxes(tickformat=".1%")
fig.show()

## 9. Seven-day rolling average

Daily reporting can be noisy because of reporting schedules and revisions. A 7-day rolling average smooths short-term fluctuations and makes the broader trajectory easier to inspect.

In [ ]:
df_global["New_Confirmed_7day_avg"] = (
    df_global["New_Confirmed"].rolling(window=7, min_periods=1).mean()
)

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(
    data=df_global,
    x="Date",
    y="New_Confirmed",
    label="Daily reported cases",
    alpha=0.35
)
sns.lineplot(
    data=df_global,
    x="Date",
    y="New_Confirmed_7day_avg",
    label="7-day rolling average",
    linewidth=2
)
plt.title("Global Daily Reported Cases and 7-Day Rolling Average")
plt.xlabel("Date")
plt.ylabel("Reported confirmed cases")
plt.tight_layout()
plt.show()

## 10. WHO region analysis

Country-level records are aggregated to WHO region and date to compare broad regional trajectories.

In [ ]:
df_who = (
    df.groupby(["WHO Region", "Date"], as_index=False)
      .agg({
          "Confirmed": "sum",
          "Deaths": "sum",
          "Recovered": "sum",
          "Active": "sum",
          "New_Confirmed": "sum",
          "New_Deaths": "sum",
          "New_Recovered": "sum"
      })
      .sort_values(["WHO Region", "Date"])
)

df_who.head()

In [ ]:
fig = px.line(
    df_who,
    x="Date",
    y="Confirmed",
    color="WHO Region",
    title="Reported Confirmed COVID-19 Cases by WHO Region",
    labels={"Confirmed": "Confirmed cases", "WHO Region": "WHO region", "Date": "Date"}
)
fig.show()

## 11. Geographic visualization

The animated map shows the reported cumulative confirmed-case count by country over time. Country names are standardized where the source naming differs from common geographic names.

In [ ]:
name_fix = {
    "US": "United States",
    "Korea, South": "South Korea",
    "Congo (Kinshasa)": "Democratic Republic of the Congo",
    "Congo (Brazzaville)": "Republic of the Congo",
    "Taiwan*": "Taiwan"
}

df_map = df.copy()
df_map["Country/Region"] = df_map["Country/Region"].replace(name_fix)

fig = px.choropleth(
    df_map,
    locations="Country/Region",
    locationmode="country names",
    color="Confirmed",
    hover_name="Country/Region",
    animation_frame="Date",
    color_continuous_scale="Reds",
    title="Reported Cumulative Confirmed Cases by Country Over Time",
    labels={"Confirmed": "Confirmed cases"}
)
fig.show()

## 12. Short-term forecasting

### Why forecast?

Forecasting is included to demonstrate a complete time-series workflow rather than to make a real-world epidemiological prediction.

The target is **global cumulative confirmed cases**. The final 14 observed days are held out as a time-ordered test set. Prophet is trained only on the earlier observations, and its predictions are evaluated against the held-out data.

A naive baseline is also included: the baseline predicts every test-day value as the last value observed in the training period. This gives the ML/forecasting model a simple benchmark to beat.

In [ ]:
from prophet import Prophet

forecast_data = df_global[["Date", "Confirmed"]].rename(
    columns={"Date": "ds", "Confirmed": "y"}
).copy()

TEST_DAYS = 14
if len(forecast_data) <= TEST_DAYS + 30:
    raise ValueError("Not enough observations for a meaningful train/test split.")

train = forecast_data.iloc[:-TEST_DAYS].copy()
test = forecast_data.iloc[-TEST_DAYS:].copy()

print("Training period:", train["ds"].min().date(), "to", train["ds"].max().date())
print("Test period:", test["ds"].min().date(), "to", test["ds"].max().date())

In [ ]:
model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=False
)
model.fit(train)

future = model.make_future_dataframe(periods=TEST_DAYS, freq="D")
forecast = model.predict(future)

pred = (
    forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
    .tail(TEST_DAYS)
    .reset_index(drop=True)
)

pred.head()

In [ ]:
# Evaluate the forecasting model against the held-out test set.
comparison = test.merge(pred, on="ds", how="left")

mae = mean_absolute_error(comparison["y"], comparison["yhat"])
rmse = np.sqrt(mean_squared_error(comparison["y"], comparison["yhat"]))

# Naive baseline: repeat the last training observation.
comparison["naive"] = train["y"].iloc[-1]
naive_mae = mean_absolute_error(comparison["y"], comparison["naive"])
naive_rmse = np.sqrt(mean_squared_error(comparison["y"], comparison["naive"]))

metrics = pd.DataFrame({
    "Model": ["Prophet", "Naive baseline"],
    "MAE": [mae, naive_mae],
    "RMSE": [rmse, naive_rmse]
})

metrics

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train["ds"], y=train["y"],
    mode="lines", name="Training data"
))
fig.add_trace(go.Scatter(
    x=test["ds"], y=test["y"],
    mode="lines+markers", name="Actual test data"
))
fig.add_trace(go.Scatter(
    x=comparison["ds"], y=comparison["yhat"],
    mode="lines+markers", name="Prophet prediction"
))
fig.add_trace(go.Scatter(
    x=comparison["ds"], y=comparison["yhat_upper"],
    mode="lines", line=dict(width=0), showlegend=False
))
fig.add_trace(go.Scatter(
    x=comparison["ds"], y=comparison["yhat_lower"],
    mode="lines", line=dict(width=0), fill="tonexty",
    name="Prediction interval"
))

fig.update_layout(
    title="14-Day Holdout Forecast: Global Cumulative Confirmed Cases",
    xaxis_title="Date",
    yaxis_title="Reported cumulative confirmed cases"
)
fig.show()

### Forecast interpretation

The forecast should be judged using the held-out test period and compared with the naive baseline. A lower MAE/RMSE indicates smaller prediction errors on this specific historical test window.

This is a **short-term statistical forecast of the reported cumulative series**, not a causal epidemiological model and not a prediction of actual infections.

## 13. Key findings

The final results below are generated directly from the cleaned dataset so that the narrative stays tied to the observed data.

In [ ]:
# Generate portfolio-ready summary statistics from the data.
start_date = df_global["Date"].min()
end_date = df_global["Date"].max()

start_confirmed = df_global.loc[df_global["Date"] == start_date, "Confirmed"].iloc[0]
end_confirmed = df_global.loc[df_global["Date"] == end_date, "Confirmed"].iloc[0]
end_deaths = df_global.loc[df_global["Date"] == end_date, "Deaths"].iloc[0]
end_recovered = df_global.loc[df_global["Date"] == end_date, "Recovered"].iloc[0]

peak_daily_cases = df_global.loc[df_global["New_Confirmed"].idxmax(), ["Date", "New_Confirmed"]]

top_country = latest.iloc[0][["Country/Region", "Confirmed"]]

summary = {
    "Observed period": f"{start_date:%d %b %Y} to {end_date:%d %b %Y}",
    "Global confirmed cases at end": int(end_confirmed),
    "Global deaths at end": int(end_deaths),
    "Global recovered at end": int(end_recovered),
    "Highest single-day reported increase": int(peak_daily_cases["New_Confirmed"]),
    "Peak daily increase date": peak_daily_cases["Date"].strftime("%d %b %Y"),
    "Top country at final date": top_country["Country/Region"],
    "Top country confirmed cases": int(top_country["Confirmed"])
}

pd.Series(summary, name="Value")

## 14. Limitations

- The dataset contains **reported** cases, deaths, and recoveries rather than a direct measurement of infections.
- Testing capacity and reporting practices differed across countries and over time.
- Retrospective corrections can create negative daily changes in a cumulative series.
- The source covers a limited historical period ending in July 2020.
- Recovery definitions and reporting completeness may differ across locations.
- Cumulative confirmed cases are non-stationary and are therefore not equivalent to daily incidence.
- Prophet's performance here is specific to the selected historical holdout window; it should not be treated as evidence of future real-world performance.

## 15. Conclusion

This project demonstrates an end-to-end workflow for working with real-world public-health data: validating the raw data, aggregating geographic records, deriving daily changes, visualizing global and regional trends, calculating descriptive indicators, and evaluating a short-term forecasting model against a time-ordered holdout set.

The main lesson is that **data quality and interpretation matter as much as visualization**. Reported COVID-19 data contain revisions and differences in reporting practices, so analytical conclusions need to be framed around what the dataset actually measures.